In [72]:
import pandas as pd
import glob
import os
import shutil
import numpy as np

In [73]:
template_path = r"EDDTemplate\ESBasic_TRC Format.xlsx"
template_df = pd.read_excel(template_path, sheet_name="ESBasic_TRC")
template_df.columns

Index(['#sys_sample_code', 'sample_name', 'sample_type_code',
       'sample_matrix_code', 'sample_date', 'sample_time', 'sys_loc_code',
       'parent_sample_code', 'start_depth', 'end_depth', 'depth_units',
       'sampling_company_code', 'samplers', 'sample_method',
       'field_filtered_flag', 'task_code', 'chain_of_custody',
       'required_turn_around_time', 'Field_SDG', 'Sample_Source',
       'lab_name_code', 'analytic_method', 'analysis_date', 'analysis_time',
       'preservative', 'LAB_PROJ_NUMBER', 'composite_yn', 'lab_sample_id',
       'Lab_SDG', 'test_type', 'test_batch_id', 'fraction', 'lab_matrix',
       'basis', 'dilution_factor', 'prep_method', 'prep_date', 'prep_time',
       'cas_rn', 'chemical_name', 'result_value', 'result_unit',
       'result_type_code', 'reportable_result', 'detect_flag',
       'lab_qualifiers', 'validator_qualifiers', 'interpreted_qualifiers',
       'validated_yn', 'method_detection_limit', 'reporting_detection_limit',
       'quantitati

In [74]:
dir = "data/granger/pfas/2025"
# Get all Excel file paths in the directory
excel_paths = glob.glob(os.path.join(dir, "*.xlsx"))
excel_paths

['data/granger/pfas/2025\\25A0027 FINAL granger 24 Jan 25 1541.xlsx',
 'data/granger/pfas/2025\\26A0032 FINAL granger 13 Jan 26 0809.xlsx']

In [75]:
filename2 = '25A0027 FINAL granger 24 Jan 25 1541.xlsx'
filename3 = '26A0032 FINAL granger 13 Jan 26 0809.xlsx'

filename = filename3

_25A0027 = pd.read_excel(f'data\\granger\\pfas\\2025\\{filename2}', sheet_name="edd")
# duplicate MW-09r
_26A0032 = pd.read_excel(f'data\\granger\\pfas\\2025\\{filename3}', sheet_name="edd")

# Check if all three dataframes have identical columns
columns_match = set(_25A0027.columns) == set(_26A0032.columns)
print(f"Do all dataframes have the same columns? {columns_match}")


Do all dataframes have the same columns? True


In [76]:
edd_mapping = {
# New EDD Column                  # EQuIS Standard Column
    "Lab_ID": "lab_sample_id",
    "Sample_ID": "sys_loc_code",  # Can also map to lab_sample_id depending on context
    "Analyte": "chemical_name",
    "Result": "result_value",  # Note: Check 'Concentration' vs 'Result' behavior below
    "Comment": "result_comments",
    "Units": "result_unit",
    "Matrix": "sample_matrix_code",
    "Sample_Collection_Date": "sample_date",
    "Date_Extracted": "prep_date",
    "Date_Analyzed": "analysis_date",
    "Detection_Limits": "method_detection_limit",
    "Reporting_Limits": "reporting_detection_limit",
    "Dilution_Factor": "dilution_factor",
    "CAS Number": "cas_rn",
    "Extraction_Method": "prep_method",
    "Sample_Fraction": "fraction",
    "Analytical_Method_Reference": "analytic_method",
    "Project_Number": "LAB_PROJ_NUMBER",
}


In [77]:
pfas_chemical_dict = {
    "11Cl-PF3OUdS": "11-chloroeosafluoroundecane-1-sulfonic acid (11Cl-PF3OUdS)",
    "11Cl-PF3OUDS": "11-chloroeicosafluoro-3-oxaundecane-1-sulfonic acid (11Cl-PF3OUDS)",
    "4:2FTS": "4:2 Fluorotelomer sulfonic acid (4:2 FTS)",
    "6:2FTS": "6:2 fluorotelomer sulfonic acid (6:2FTS)",
    "8:2FTS": "8:2 fluorotelomer sulfonic acid (8:2FTS)",
    "9Cl-PF3ONS": "9-chlorohexadecafluoro-3-oxanonane-1-sulfonic acid (9Cl-PF3ONS)",
    "ADONA": "4,8-dioxa-3H-perfluorononanoic acid (ADONA)",
    "HFPO-DA": "Hexafluoropropylene oxide dimer acid (HFPO-DA)",
    "PFBA": "Perfluorobutanoic acid (PFBA)",
    "PFBS": "Perfluorobutanesulfonic acid (PFBS)",
    "PFDA": "Perfluorodecanoic acid (PFDA)",
    "PFDoA": "Perfluorododecanoic acid (PFDoA)",
    "PFDS": "Perfluorodecanesulfonic acid (PFDS)",
    "PFHpA": "Perfluoroheptanoic acid (PFHpA)",
    "PFHpS": "Perfluoroheptanesulfonic acid (PFHpS)",
    "PFHxA": "Perfluorohexanoic acid (PFHxA)",
    "PFHXA": "Perfluorohexanoic acid (PFHxA)",
    "PFHxS": "Perfluorohexanesulfonic acid (PFHxS)",
    "PFNA": "Perfluorononanoic acid (PFNA)",
    "PFNS": "Perfluorononanesulfonic acid (PFNS)",
    "PFOA": "Perfluorooctanoic acid (PFOA)",
    "PFOS": "Perfluorooctanesulfonic acid (PFOS)",
    "PFOSA": "Perfluorooctanesulfonamide (PFOSA)",
    "PFPeA": "Perfluoropentanoic acid (PFPeA)",
    "PFPeS": "Perfluoropentanesulfonic acid (PFPeS)",
    "PFTeDA": "Perfluorotetradecanoic acid (PFTeDA)",
    "PFTrDA": "Perfluorotridecanoic acid (PFTrDA)",
    "PFUnA": "Perfluoroundecanoic acid (PFUnA)",
    "NMeFOSAA": "N-Methyl perfluorooctanesulfonamidoacetic acid  (NMeFOSAA)",
    "NEtFOSAA": "N-Ethyl perfluorooctanesulfonamidoacetic acid (NEtFOSAA)",

    "13C4-PFBA": "13C4-Perfluorobutanoic acid (13C4-PFBA)",
    "13C5-PFPEA": "13C5-Perfluoropentanoic acid (13C5-PFPeA) (Surr)",
    "13C5-PFHXA": "13C5-Perfluorohexanoic acid (13C5-PFHxA) (Surr)",
    "13C4-PFHPA": "13C4-Perfluoroheptanoic acid (13C4-PFHpA) (Surr)",
    "13C8-PFOA": "13C8-Perfluorooctanoic acid (13C8-PFOA) (Surr)",
    "13C9-PFNA": "13C9-Perfluorononanoic acid (13C9-PFNA) (Surr)",
    "13C6-PFDA": "13C6-Perfluorodecanoic acid (13C6-PFDA) (Surr)",
    "13C7-PFUnA": "13C7-Perfluoroundecanoic acid (13C7-PFUnDA) (Surr)",
    "13C2-PFDOA": "13C2-Perfluorododecanoic acid (13C2-PFDoDA) (Surr)",
    "13C2-PFTEDA": "13C2-Perfluorotetradecanoic acid (13C2-PFTeDA) (Surr)",
    "13C3-PFBS": "13C3-Perfluorobutane sulfonic acid (13C3-PFBS) (Surr)",
    "13C3-PFHXS": "13C3-Perfluorohexane sulfonic acid (13C3-PFHxS) (Surr)",
    "13C8-PFOS": "13C8-Perfluorooctane sulfonic acid (13C8-PFOS) (Surr)",
    "13C2-4:2FTS": "13C2-4:2 Fluorotelomer sulfonic acid (13C2-4:2 FTS)",
    "13C2-6:2FTS": "13C2-6:2 Fluorotelomer sulfonic acid (13C2-6:2 FTS)",
    "13C2-8:2FTS": "13C2-8:2 Fluorotelomer sulfonic acid (13C2-8:2 FTS)",
    "13C8-PFOSA": "13C8-Perfluorooctane sulfonamide (13C8-FOSA) (Surr)",
    "D3-NMEFOSAA": "d3-N-Methyl perfluorooctane sulfonamido acetic acid (d3-NMeFOSAA) (Surr)",
    "D5-NETFOSAA": "d5-2-(N-ethyl perfluorooctane sulfonamido) acetic acid (d5-NEtFOSAA) (Surr)",
    "13C3-HFPO-DA": "13C3-Hexafluoropropylene oxide dimer acid (13C3-HFPO-DA) (Surr)",


}

In [78]:
fix_pfas_cas = {"N-Methyl perfluorooctanesulfonamidoacetic acid  (NMeFOSAA)": "2355-31-9",
            "N-Ethyl perfluorooctanesulfonamidoacetic acid (NEtFOSAA)" : "2991-50-6",
            "13C7-Perfluoroundecanoic acid (13C7-PFUnDA) (Surr)": "13C7-PFUnA",
            "13C2-Perfluorododecanoic acid (13C2-PFDoDA) (Surr)": "13C2-PFDoDA",
            "13C2-4:2 Fluorotelomer sulfonic acid (13C2-4:2 FTS)": "13C2-4:2 FTS",
            "13C2-6:2 Fluorotelomer sulfonic acid (13C2-6:2 FTS)": "13C2-6:2 FTS",
            "13C2-8:2 Fluorotelomer sulfonic acid (13C2-8:2 FTS)": "13C2-8:2 FTS",
            "13C8-Perfluorooctane sulfonamide (13C8-FOSA) (Surr)": "13C8-FOSA",
            }

In [79]:
edd_to_db_methods = {
    'EPA 200.8 Rev. 5.4': 'E200.8',
    'EPA 200.7 Rev. 4.4': 'E200.7',
    'EPA 8260D': 'SW8260D',
    'EPA 6020B': 'SW6020B',
    'Calculation': 'CALC',
    'EPA 350.1 Rev. 2.0': 'E350.1',
    'SM 2540 C-20': 'SM-2540',
    'SM 2320 B-21': 'SM2320',
    'SM 5310 B-14': 'SM5310',
    'EPA 300.0 Rev. 2.1': 'E300.0',
    'SM 4500-Cl D-21': 'SM4500',
    'EPA 410.4 Rev. 2.0': 'E410.4',
    'EPA 537M': 'E537 Mod'
}

In [80]:
sample_id_dict = {
    "Under Drain": "UNDERDRAIN",
    "Leachate": "LEACHATE",
    'Groesbeck Pond': 'GROESBECK POND',
    'Cooper 1': 'COOPER-1',
    'Cooper 2': 'COOPER-2',
    'MW-3sr': 'MW-03sr',
    'MW-3dr': 'MW-03dr',
    'MW-5sr': 'MW-05sr',
    'MW-5dr': 'MW-05dr',
    'MW-6dr': 'MW-06dr',
    'MW-6r2': 'MW-06r2',
    'MW-6sr': 'MW-06sr',
    "MW-9r" : "MW-09r",
    "P28r" : "P-28r",
}

In [81]:
lab_matrix_dict = {
    'Ground Water': 'WG',
    'Aqueous': 'WU',
    'Water': 'WU',
}

In [82]:
def assign_matrix(sample_type_code):
    sample_str = str(sample_type_code).strip()
    
    # 1. Groundwater wells starting with MW- and P-
    if sample_str.startswith("N"): 
        return 'WG'  # Groundwater
    elif sample_str.startswith("FD"):
        return 'WG'
    elif sample_str.startswith("FB"):
        return 'WQ'
    elif sample_str.startswith("EB"):
        return 'WQ'
    
    else:
        return 'U' # Unknown/Unassigned

In [83]:
# Rename columns to match the EDD template
# SAMPDATA = _24A0011
# QCDATA = _24A0011QC.rename(columns=qc_edd_mapping)

# # Reindex df_new using the exact column structure and order of df_template
# SAMPDATA = SAMPDATA.reindex(columns=template_df.columns)
# QCDATA = QCDATA.reindex(columns=template_df.columns)

# EDD_TO_PROCESS = pd.concat([SAMPDATA, QCDATA], ignore_index=True)
EDD_TO_PROCESS = _26A0032.copy()


# Add required column Lab_SDG
lab_sgd = '26A0032' ##############################

# Preprocessing
# 1. Fill NaN values in 'Prefix' with an empty string so concatenation doesn't break
EDD_TO_PROCESS['Prefix'] = EDD_TO_PROCESS['Prefix'].fillna('').astype(str).str.strip()
EDD_TO_PROCESS['Result'] = EDD_TO_PROCESS['Result'].fillna('').astype(str).str.strip()

# 2. Combine Prefix and Result into your template's 'result_value'
#    (Adds a space between them only if a prefix exists)
EDD_TO_PROCESS['Result'] = EDD_TO_PROCESS.apply(
    lambda row: f"{row['Prefix']} {row['Result']}".strip() if row['Prefix'] else row['Result'], 
    axis=1
)
EDD = EDD_TO_PROCESS.rename(columns=edd_mapping)
EDD = EDD.reindex(columns=template_df.columns)

In [84]:
EDD_TO_PROCESS.Sample_Collection_Date.unique()

<StringArray>
['01/06/2026']
Length: 1, dtype: str

In [85]:
EDD_TO_PROCESS.Sample_ID.unique()

<StringArray>
['MW-19r', 'MW-9r', 'DUPLICATE', 'EQUIPMENT BLANK', 'FIELD BLANK']
Length: 5, dtype: str

In [86]:
EDD.result_value.unique()

<StringArray>
[ '< 3.2',  '< 0.8',   '21.0',   '10.0',    '9.0',   '14.0',   '17.0',
  '170.0',  '140.0',   '95.0',   '15.0',    '5.5',  '< 3.3', '< 0.82',
   '11.0',    '2.7',    '3.2',    '6.9',    '2.6',    '1.7',   '12.0',
    '7.3',    '3.0',    '3.5',    '2.8',    '1.6',   '13.0',    '6.7',
  '< 1.6']
Length: 29, dtype: str

In [87]:
# 1. Handle Datetimes safely
temp_datetime = pd.to_datetime(EDD["sample_date"], errors='coerce')
EDD["sample_date"] = temp_datetime.dt.strftime("%m/%d/%Y")  # 04/13/2026
EDD["sample_time"] = temp_datetime.dt.strftime("%H:%M:%S")  # 09:10:00

# 1. Handle Datetimes safely
temp_datetime2 = pd.to_datetime(EDD["analysis_date"], errors='coerce')
EDD["analysis_date"] = temp_datetime2.dt.strftime("%m/%d/%Y")  # 04/13/2026
# EDD["analysis_time"] = temp_datetime2.dt.strftime("%H:%M:%S")  # 09:10:00

# 1. Handle Datetimes safely
temp_datetime3 = pd.to_datetime(EDD["prep_date"], errors='coerce')
EDD["prep_date"] = temp_datetime3.dt.strftime("%m/%d/%Y")  # 04/13/2026
EDD["prep_time"] = temp_datetime3.dt.strftime("%H:%M:%S")  # 09:10:00

# 2. Preserve sample_name BEFORE modifying sys_loc_code, then assign matrix
EDD['sample_name'] = EDD['sys_loc_code']

# mappings
# update the chemical_name column using the pfas chemical dict
EDD["chemical_name"] = EDD["chemical_name"].map(pfas_chemical_dict).fillna(EDD["chemical_name"])
# update some of the cas numbers based on fix_pfas_cas dictionary
EDD["cas_rn"] = EDD["chemical_name"].map(fix_pfas_cas).fillna(EDD["cas_rn"]).fillna("Unknown TIC")
EDD['analytic_method'] = EDD['analytic_method'].map(edd_to_db_methods)



EDD['lab_matrix'] = EDD['lab_matrix'].map(lab_matrix_dict)

# 3. Clean QC samples out of sys_loc_code without altering text casing globally
######################################################################################
blank_these_locs = ["equipment blank", "field blank a", "field blank b", "trip blank", "field blank", "blank"]
######################################################################################
is_qc_blank = EDD["sys_loc_code"].astype(str).str.strip().str.lower().isin(blank_these_locs)
EDD.loc[is_qc_blank, "sys_loc_code"] = ""

# 4. Generate #sys_sample_code safely
date_yyyymmdd = pd.to_datetime(EDD["sample_date"], errors="coerce").dt.strftime("%Y%m%d").fillna("")
EDD['#sys_sample_code'] = EDD['sample_name'].str.strip() + "_" + date_yyyymmdd

# Skipping Duplicate sample code automation for now
# NO duplcates in #sys_sample_code, but if there were, we would handle them like this:
# 5. Define Duplicate Hierarchies
# duplicate_01_ID, parent_01_sample_id = "Duplicate A", "MW-19r"
# # duplicate_02_ID, parent_02_sample_id = "Duplicate B", "MW-20r"
# # duplicate_03_ID, parent_03_sample_id = "Duplicate C", "MW-41d"

# # # Create clean conditional masks
# cond_01 = EDD['sample_name'].astype(str).str.strip().str.upper() == duplicate_01_ID.upper()
# # cond_02 = EDD['sample_name'].astype(str).str.strip().str.upper() == duplicate_02_ID.upper()
# # cond_03 = EDD['sample_name'].astype(str).str.strip().str.upper() == duplicate_03_ID.upper()

# cond_01b = EDD['sample_name'].astype(str).str.strip().str.upper() == parent_01_sample_id.upper()
# # cond_02b = EDD['sample_name'].astype(str).str.strip().str.upper() == parent_02_sample_id.upper()
# # cond_03b = EDD['sample_name'].astype(str).str.strip().str.upper() == parent_03_sample_id.upper()

# # # FIX: Use the existing robust masks to update sys_loc_code safely
# EDD.loc[cond_01, "sys_loc_code"] = parent_01_sample_id

# # Map Parent Sample Codes
# EDD.loc[cond_01, 'parent_sample_code'] = EDD.loc[cond_01b, "#sys_sample_code"][EDD.loc[cond_01b, "#sys_sample_code"].first_valid_index()]
# EDD.loc[cond_02, 'parent_sample_code'] = EDD.loc[cond_02b, '#sys_sample_code'][EDD.loc[cond_02b, '#sys_sample_code'].first_valid_index()]
# EDD.loc[cond_03, 'parent_sample_code'] = EDD.loc[cond_03b, '#sys_sample_code'][EDD.loc[cond_03b, '#sys_sample_code'].first_valid_index()]



EDD['sys_loc_code'] = EDD['sys_loc_code'].map(sample_id_dict).fillna(EDD['sys_loc_code'])  # Map to standardized sample names if possible, else keep original code

# 6. FIX: Use np.select to evaluate sample types conditionally without overwriting
sample_name_lower = EDD["sample_name"].astype(str).str.lower()
conditions = [
    sample_name_lower.str.contains("equipment"),
    sample_name_lower.str.contains("trip"),
    sample_name_lower.str.contains("duplicate"),
    sample_name_lower.str.contains("field")
]
choices = ["EB", "TB", "FD", "FB"]
EDD['sample_type_code'] = np.select(conditions, choices, default="N")

EDD['sample_matrix_code'] = EDD['sample_type_code'].apply(assign_matrix)

res_str = EDD['result_value'].astype(str)
# 7. Data normalization
# Define your "Non-Detect" conditions
conditions = [
    EDD['result_value'].isna(),
    res_str == "ND",
    res_str.str.contains('<', na=False),
    EDD['result_value'] == pd.NaT
]

# 3. Apply the choices: "N" if any condition is met, otherwise "Y"
EDD['detect_flag'] = np.select(conditions, ["N"] * len(conditions), default="Y")
EDD['result_value'] = EDD['result_value'].apply(lambda x: None if x in ["ND", pd.NaT] or pd.isnull(x) else x)
EDD['result_unit'] = EDD['result_value'].apply(lambda x: None if pd.isnull(x) else "ng/L") # Note: Ensure your DB allows null units for NDs
EDD['detection_limit_unit'] = EDD['reporting_detection_limit'].apply(lambda x: None if pd.isnull(x) else "ng/L") # Note: Ensure your DB allows null units for NDs
# if result value is "<" or "" set detect_flag to "N" and result_value to ""
EDD.loc[EDD['result_value'] == "<", 'result_value'] = None
EDD.loc[EDD['result_value'] == None, 'result_unit'] = None
EDD.loc[EDD['result_value'].isnull(), 'detect_flag'] = "N"


# In one line, fix the dilution factor column to not exceed 1 decimal place, but also not convert integers to floats unnecessarily
EDD['dilution_factor'] = EDD['dilution_factor'].round(1).where(EDD['dilution_factor'].notnull(), None)

# 8. FIX: Reindex at the very end to ensure temporary calculation spaces aren't mutated prematurely
EDD = EDD.reindex(columns=template_df.columns)

# Addl req'd columns to be filled in:
EDD['detect_flag'] = EDD['result_value'].apply(lambda x: "N" if pd.isnull(x) or x == "ND" or x == pd.NaT else "Y")
EDD['result_value'] = EDD['result_value'].apply(lambda x: None if x in ["ND", pd.NaT] or pd.isnull(x) else x)
EDD['result_unit'] = EDD['result_value'].apply(lambda x: None if pd.isnull(x) else "ng/L")
EDD['result_type_code'] = "TRG"
# Final Fill-Ins for Required Fields
EDD['result_type_code'] = "TRG"
EDD['reportable_result'] = "Yes"
EDD['test_type'] = "Initial"
EDD['Lab_SDG'] = lab_sgd
EDD['prep_method'] = "Method"

EDD['test_type'] = "Initial"
EDD['fraction'] = EDD['fraction'].fillna("N")


In [88]:
EDD.sys_loc_code.unique()

<StringArray>
['MW-19r', 'MW-09r', 'DUPLICATE', '']
Length: 4, dtype: str

In [89]:
# 2. Make an exact copy of the template file
# Ouptut path for the new EDD file to be created
# Make a unique name for the output file by including the original file name and a timestamp
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
output_path = f"{dir}\\{filename.split('.')[0]}_ESBasic_{timestamp}.xlsx"
shutil.copy(template_path, output_path)

# 3. Write your dataframe into the copied template
# (Using 'a' mode allows you to append/write data to an existing sheet)
with pd.ExcelWriter(
    output_path, engine="openpyxl", mode="a", if_sheet_exists="overlay"
) as writer:
    EDD.to_excel(
        writer,
        sheet_name="ESBasic_TRC",  # <-- Change this to the exact name of the sheet in your template
        index=False,
        header=False,  # <-- Set to False if your template already has the headers typed out
        startrow=2,  # <-- Starts writing on row 2 (0-indexed), assuming row 1 has your headers
    )